In [4]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from getpass import getpass
import os

In [7]:
if not os.getenv("OPENAI_API_KEY_EMBEDDINGS"):
    os.environ["OPENAI_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model: ")

if not os.getenv("OPENAI_API_KEY_LLM"):
    os.environ["OPENAI_API_KEY_LLM"] = getpass("Enter your RCP API key for LLM model: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_EMBEDDINGS")
)

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name="basic_pdf_split_pages",
    url="http://localhost:6333",
    #retrieval_mode= TODO : tester hybrid
)

llm = OpenAI(
    model="mistralai/Mistral-Small-3.2-24B-Instruct-2506",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_LLM")
)

In [9]:
retriever = qdrant.as_retriever(search_type="similarity", search_kwargs={"k": 2}) # TODO : gerer les parametres

prompt_template = """
You are an intelligent assistant tasked with answering user queries based on provided context.
Use the following context to respond to the user's question in the language of the question.

Context:
{context}

Question:
{query}

Answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query_example = "Comment les responsabilités en matière de contrôle interne sont-elles réparties entre la direction, les unités opérationnelles et les fonctions de supervision (audit, contrôle des risques) au sein de l'EPFL?"

In [11]:
response = chain.invoke(query_example)

In [12]:
response

" Based on the provided context, the documents do not contain specific information about how responsibilities for internal control are distributed among the management, operational units, and supervisory functions (audit, risk control) at EPFL.\n\nTo get accurate information on this topic, you may need to refer to the detailed internal control framework or organizational policies of EPFL, which are not included in the given context. You could also consult the EPFL's official regulations, such as the **Règlement Financier EPFL (LEX 5.1.1, Version 1.12 du 1er janvier 2025)** or other governance documents that outline internal control responsibilities.\n\nWould you like assistance in finding additional resources or searching for relevant EPFL documents?"